In [3]:
#Rashi Pande
#23070521177
import numpy as np

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.model_selection import train_test_split


# ==================================================
# 1. CREATE DATASET
# ==================================================

sentences = [
    "book a flight from mumbai to delhi",
    "book a ticket from nagpur to pune",
    "find a flight from delhi to bangalore",
    "travel from mumbai to chennai",
    "flight from pune to hyderabad",
    "book flight from chennai to mumbai",
    "i want to fly from nagpur to delhi",
    "find ticket from bangalore to mumbai",
    "book a flight from hyderabad to pune",
    "travel from delhi to chennai",
    "i need a flight from pune to delhi",
    "book ticket from mumbai to nagpur",
    "find flight from chennai to bangalore",
    "travel from hyderabad to mumbai",
    "book a flight from delhi to pune",
    "i want to travel from nagpur to mumbai",
    "find a ticket from pune to chennai",
    "book flight from bangalore to delhi",
    "travel from mumbai to hyderabad",
    "find flight from chennai to pune"
]


# ==================================================
# 2. SLOT LABELS
# ==================================================

labels = [
    "O O O O B-from_city O B-to_city",
    "O O O O B-from_city O B-to_city",
    "O O O O B-from_city O B-to_city",
    "O O O B-from_city O B-to_city",
    "O O B-from_city O B-to_city",
    "O O O B-from_city O B-to_city",
    "O O O O O B-from_city O B-to_city",
    "O O O B-from_city O B-to_city",
    "O O O O B-from_city O B-to_city",
    "O O B-from_city O B-to_city",
    "O O O O O B-from_city O B-to_city",
    "O O O B-from_city O B-to_city",
    "O O O B-from_city O B-to_city",
    "O O B-from_city O B-to_city",
    "O O O O B-from_city O B-to_city",
    "O O O O O B-from_city O B-to_city",
    "O O O O B-from_city O B-to_city",
    "O O O B-from_city O B-to_city",
    "O O B-from_city O B-to_city",
    "O O O B-from_city O B-to_city"
]


# ==================================================
# 3. TOKENIZATION
# ==================================================

tokenizer = Tokenizer()

tokenizer.fit_on_texts(sentences)

X = tokenizer.texts_to_sequences(sentences)


# ==================================================
# 4. ENCODE LABELS
# ==================================================

label_to_id = {
    "O": 0,
    "B-from_city": 1,
    "B-to_city": 2
}

y = []

for label_sentence in labels:

    label_ids = [
        label_to_id[label]
        for label in label_sentence.split()
    ]

    y.append(label_ids)


# ==================================================
# 5. PADDING
# ==================================================

max_len = max(len(seq) for seq in X)

X = pad_sequences(
    X,
    maxlen=max_len,
    padding="post"
)

y = pad_sequences(
    y,
    maxlen=max_len,
    padding="post",
    value=0
)


# ==================================================
# 6. TRAIN-TEST SPLIT
# ==================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# ==================================================
# 7. BUILD LSTM MODEL
# ==================================================

vocab_size = len(tokenizer.word_index) + 1
num_labels = len(label_to_id)

model = Sequential()

model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=32,
        input_length=max_len
    )
)

model.add(
    LSTM(
        64,
        return_sequences=True
    )
)

model.add(
    Dense(
        num_labels,
        activation="softmax"
    )
)


# ==================================================
# 8. COMPILE MODEL
# ==================================================

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


# ==================================================
# 9. DISPLAY MODEL
# ==================================================

model.summary()


# ==================================================
# 10. TRAIN MODEL
# ==================================================

history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=4,
    validation_split=0.2,
    verbose=1
)


# ==================================================
# 11. EVALUATE MODEL
# ==================================================

loss, accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("\nTest Accuracy:", accuracy)


# ==================================================
# 12. TEST NEW SENTENCE
# ==================================================

test_sentence = "book a flight from mumbai to pune"

test_sequence = tokenizer.texts_to_sequences(
    [test_sentence]
)

test_sequence = pad_sequences(
    test_sequence,
    maxlen=max_len,
    padding="post"
)


# ==================================================
# 13. PREDICTION
# ==================================================

prediction = model.predict(
    test_sequence,
    verbose=0
)

predicted_ids = np.argmax(
    prediction,
    axis=-1
)[0]


# ==================================================
# 14. CONVERT IDs TO LABELS
# ==================================================

id_to_label = {
    0: "O",
    1: "B-from_city",
    2: "B-to_city"
}

words = test_sentence.split()

print("\nSlot Filling Result:")

for word, label_id in zip(
    words,
    predicted_ids
):

    print(
        word,
        "->",
        id_to_label[label_id]
    )

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 273ms/step - accuracy: 0.5729 - loss: 1.0932 - val_accuracy: 0.7500 - val_loss: 1.0851
Epoch 2/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.7500 - loss: 1.0763 - val_accuracy: 0.7500 - val_loss: 1.0712
Epoch 3/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.7500 - loss: 1.0572 - val_accuracy: 0.7500 - val_loss: 1.0550
Epoch 4/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.7500 - loss: 1.0334 - val_accuracy: 0.7500 - val_loss: 1.0357
Epoch 5/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.7500 - loss: 1.0032 - val_accuracy: 0.7500 - val_loss: 1.0123
Epoch 6/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.7500 - loss: 0.9641 - val_accuracy: 0.7500 - val_loss: 0.9852
Epoch 7/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.7500 - loss: 0.9161 - val_accuracy: 0.7500 - val_loss: 0.9578
Epoch 8/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.7500 - loss: 0.8658 - val_accuracy: 0.7500 - val_loss